# Parte 1 - RAG - Retrieval-Augmented Generation

### Objetivo: Desenvolver um sistema que responde perguntas sobre um conjunto de artigos científicos locais (PDFs), usando uma abordagem de Retrieval-Augmented Generation.






#ATIVIDADE

- Altere os pontos marcados com #TODO(tópico 5 e 6)
- Carregar seus próprios artigos e datasets em PDF (tópico 2).
- Usar modelo gratuito (nossa sugestão é o llama via groq).
- Avaliar respostas automaticamente com métricas de NLP.

**Observação 01:** cada aluno deve adaptar o código a um domínio específico da sua linha de pesquisa (ex: Engenharia de software, IHC, IA, robótica, etc) e comparar a performance.


**Observação 02:** Caso necessário, faça suas alterações no código, conforme os conceitos vistos em sala de aula, para adequar ao caso específico que esteja tratando.

In [ ]:
!pip install PyPDF2

# 1. Upload dos PDFs
Você carregará os PDFs que gostaria que fossem analisados.

In [ ]:
from google.colab import files
uploaded = files.upload()

import os
from PyPDF2 import PdfReader

# Cria pasta para os PDFs
os.makedirs("corpus", exist_ok=True)
for fname in uploaded.keys():
    os.rename(fname, os.path.join("corpus", fname))

print("PDFs carregados:", os.listdir("corpus"))

# 2. Leitura e extração do texto dos PDFs
A função abaixo irá gerar o corpus (que é uma lista de textos). Cada elemento do corpus é o texto de um PDF carregado anteriormente.

In [ ]:
def load_papers(folder):
    corpus = []
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            reader = PdfReader(os.path.join(folder, file))
            text = " "
            for page in reader.pages:
                text = page.extract_text() or " "
                corpus.append(text)
    return corpus

texts = load_papers("corpus")
print(f"{len(texts)} chunks carregados.")

# 3. Embeddings - Criação
Usar o sentence-transformers para transformar os textos extraídos dos PDFs em embeddings. (Se colab pedir acesso, conceda)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, convert_to_tensor=True)

# 4. Função de Recuperação - (R)AG
### Implementar o mecanismo de busca vetorial. Aqui entra o retriever: busca semântica por similaridade de embeddings.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(query, texts, embeddings, top_k=2, max_chars=3000):
    """
    Recupera os textos mais relevantes limitando o tamanho total (max_chars)
    para não exceder o limite de tokens do modelo Groq.
    """
    query_emb = model.encode([query])
    scores = cosine_similarity(query_emb, embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    total_len = 0
    for i in top_indices:
        snippet = texts[i]
        if total_len + len(snippet) > max_chars:
            snippet = snippet[: max_chars - total_len]  # corta para caber no limite
        results.append(snippet)
        total_len += len(snippet)
        if total_len >= max_chars:
            break
    return results

# 5. Integrar com uma LLM - R(AG)

### O conteúdo recuperado é passado como contexto ao modelo llama-3.3:70b a partir do groq

In [ ]:
!pip install --upgrade langchain langchain-core langchain-groq


In [ ]:
from langchain_groq import ChatGroq

#TODO sigas os passos em https://groq.com/ para pegar sua chave: Developers -> Free API key
GROQ_API_KEY = input( "Cole sua chave Groq aqui e dê <enter>: ")

client = ChatGroq(
    model=" ",  #TODO modelo gratuito
    api_key=GROQ_API_KEY,
    temperature=0.2
)

def generate_answer(query, context):
    prompt = f"""
Use o contexto abaixo para responder a pergunta com precisão científica.
Contexto: {context}
Pergunta: {query}
"""
    response = client.invoke(prompt).content
    return response

# 6. Teste do modelo

In [ ]:

query = #TODO " Altere a query para conter a sua pergunta que faça sentido no contexto dos PDFs anexados"
context = " ".join(retrieve(query, texts, embeddings))
answer = generate_answer(query, context)
print("\nResposta gerada:\n", answer)

# Parte 2 - Pesquisa na web - Web-based RAG ou Online RAG
Nesta seção, faremos uma prática de buscas de informações na web. O objetivo é dar ao modelo dados atualizados retirados de artigos na web. Neste exemplo, no Retrieval **(Recuperação)**, ao invés de buscarmos de um corpus de documentos ou banco de dados, buscaremos da web. O conteúdo extraído da página da web será usado para Aumentar **(Augment)** o prompt fornecido ao modelo. E por fim, o modelo usará o prompt para Gerar **(Generation)** uma resposta que será o resumo de um artigo.

Em resumo, iremos:
demonstrar um fluxo de RAG (Retrieval Augmented Generation) buscando informações na internet usando DuckDuckGo, extraindo o conteúdo de um artigo relevante e resumindo-o usando o LLM pelo Groq.

#ATIVIDADE

**Altere os pontos marcados com #TODO** no código para testar diferentes
resultados.

Por exemplo, se houver: QUANT_MAX_ARTIGOS = 5  # TODO
mude para 10, por exemplo e veja como muda a seleção de artigos.

**Escolha o artigo que será usado.**

Por padrão, search_results[0] pega apenas o primeiro. Você pode testar com outro índice para ver respostas diferentes.

**Volte à Parte 1 e repita a execução.**

Lá o código junta os resultados do retrieve() nos chunks e passa esse contexto para o LLM.

Assim, você consegue comparar como as alterações influenciam a resposta final do modelo.





## 1. Instalar bibliotecas necessárias

Instalar bibliotecas para buscar na web (DuckDuckGo) e para extrair o conteúdo de páginas web.


In [ ]:
!pip install ddgs
!pip install beautifulsoup4
!pip install requests

##2. Realizar busca na web

Usar a ferramenta de busca para encontrar artigos relevantes com base em uma consulta do usuário. O duckduckgo (ddgs) faz o trabalho de buscar artigos na Internet, assim como o Google.


In [ ]:
from ddgs import DDGS

ddgs = DDGS()
QUANT_MAX_ARTIGOS = 5

#TODO altere a query abaixo para o contexto que deseja. Use a sintaxe como a do exemplo colocado na query
query = "LLM + artificial intelligence"
search_results = ddgs.text(query, max_results=QUANT_MAX_ARTIGOS)

print("Resultados da busca:")
for result in search_results:
    print(f"Título: {result['title']}")
    print(f"URL: {result['href']}")
    print(f"Descrição: {result['body']}\n")

##3. Extrair conteúdo do artigo

Acessar a URL do artigo retornado pela busca e extrair o texto principal.


In [ ]:
import requests
from bs4 import BeautifulSoup

article_text = None
if not search_results:
    print("Nenhuma URL encontrada para extração.")
else:
    #TODO altere search_results[0] para escolher aleatoriamente o artigo de 0 a QUANT_MAX_ARTIGOS-1
    #TODO é possível que os sites não permitam acessar o conteúdo, dando erro. Altere de search_results[0] para algum índice de site diferente de 0\
    #     para tentar baixar conteúdo de algum dos sites baixados.
    article_url = search_results[0]['href']

    try:
        response = requests.get(article_url, timeout=10) # Adicionado timeout para evitar travamentos
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            # Tentativa de encontrar o texto principal. Isso pode precisar de ajustes
            # dependendo da estrutura HTML dos sites.
            article_text = ""
            paragraphs = soup.find_all('p')
            for p in paragraphs:
                article_text += p.get_text() + "\n"

            if article_text:
                print(f"Conteúdo do artigo extraído da URL: {article_url}")
                print("Primeiros 500 caracteres do texto extraído:")
                print(article_text[:500])
            else:
                print(f"Não foi possível extrair texto principal da URL: {article_url}")

        else:
            print(f"Erro ao acessar a URL {article_url}. Código de status: {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"Erro ao acessar a URL {article_url}: {e}")

print(len(article_text))

##4. Quebrando os chunks

Quebrar em chunks o texto extraído para utilizar com contexto. Utilizando outra abordagem.


In [ ]:
!pip install langchain langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configura o Text Splitter
# Instancia o separador de texto com as suas especificações
text_splitter = RecursiveCharacterTextSplitter(
    # Tamanho máximo de cada chunk em caracteres
    chunk_size=1500,
    # Tamanho de sobreposição entre chunks.
    # Isso ajuda a manter o contexto entre os chunks adjacentes.
    chunk_overlap=250,
    # Separadores que o splitter tentará usar, em ordem:
    # 1. Parágrafos (\n\n)
    # 2. Novas linhas (\n)
    # 3. Espaços (' ')
    # 4. Caracteres vazios ('')
    separators=["\n\n", "\n", " ", ""],
    length_function=len # Função usada para medir o tamanho (len para caracteres)
)

# 3. Quebrar o texto
chunks = text_splitter.create_documents([article_text])

# 4. Imprimir os resultados para verificação

print(f"Número total de Chunks criados: **{len(chunks)}**\n")
print("-" * 50)

# Itera sobre os chunks (objetos Document do LangChain)
for i, chunk in enumerate(chunks):
    content = chunk.page_content
    print(f"*** CHUNK {i+1} (Tamanho: {len(content)} caracteres) ***")
    print(content)
    print("-" * 50)